# 04 — Testes do Agente (≥ 5 Cenários)  
**OscaBet Agent** · Validação end-to-end do `predictor.py`

Carrega o modelo treinado e simula o `run_prediction_engine()`  
para os 5 cenários exigidos na entrega.


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import json
from pathlib import Path

# ──────────────────────────────────────────────────────────────────────────────
BASE_DIR   = Path("../..").resolve()
PROC_DIR   = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "agent" / "models"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── Carregar metadados e features ─────────────────────────────────────────────
with open(MODELS_DIR / "oscabet_nn_v1_meta.json") as f:
    meta = json.load(f)

features_df = pd.read_csv(PROC_DIR / "features.csv", parse_dates=["date"])
targets_df  = pd.read_csv(PROC_DIR / "targets.csv",  parse_dates=["match_date"])

FEAT_COLS = meta["feat_cols"]
WINDOW    = meta.get("hyperparams", {}).get("window", 10)

print(f"Features esperadas pelo modelo: {len(FEAT_COLS)}")
print(f"Meta do treinamento: cutoff={meta['train_cutoff']}, n_train={meta['n_train']}")


Features esperadas pelo modelo: 69
Meta do treinamento: cutoff=2026-05-01, n_train=15877


In [ ]:
# ── Recarregar arquitetura ─────────────────────────────────────────────────────
class OscaBetNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.30),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(128, 64),                             nn.ReLU(),
        )
        self.head_result  = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 3))
        self.head_yellow  = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 3))
        self.head_corners = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 3))

    def forward(self, x):
        h = self.backbone(x)
        return self.head_result(h), self.head_yellow(h), self.head_corners(h)

checkpoint = torch.load(MODELS_DIR / "oscabet_nn_v1.pt", map_location=DEVICE, weights_only=False)
model = OscaBetNN(input_dim=meta["input_dim"]).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("✅ Modelo carregado com sucesso")

softmax = nn.Softmax(dim=1)


In [4]:
def predict_match(home_team: str, away_team: str, league: str,
                  features_df: pd.DataFrame) -> dict:
    """
    Simula o run_prediction_engine() do tools.py.
    Busca as features mais recentes dos dois times e retorna as probabilidades.
    """
    # Busca última partida de cada time para obter suas features recentes
    def last_feats(team, side, league):
        mask = (features_df["league"] == league) & (features_df[f"{side}_team"] == team)
        sub  = features_df[mask].sort_values("date")
        if len(sub) == 0:
            return None
        return sub.iloc[-1]

    home_row = last_feats(home_team, "home", league)
    away_row = last_feats(away_team, "away", league)

    if home_row is None or away_row is None:
        return {"error": f"Time não encontrado: {home_team if home_row is None else away_team}"}

    # Monta o vetor de features mesclando home e away
    x_vals = []
    for col in FEAT_COLS:
        if col.startswith("home_") and col in home_row.index:
            x_vals.append(home_row[col])
        elif col.startswith("away_") and col in away_row.index:
            x_vals.append(away_row[col])
        elif col.startswith("h2h_") and col in home_row.index:
            x_vals.append(home_row[col])
        elif col.startswith("league_") and col in home_row.index:
            x_vals.append(home_row[col])
        else:
            x_vals.append(0.5)  # fallback neutro

    x_tensor = torch.tensor([x_vals], dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        lr, ly, lc = model(x_tensor)
        pr = softmax(lr).cpu().numpy()[0]
        py = softmax(ly).cpu().numpy()[0]
        pc = softmax(lc).cpu().numpy()[0]

    result_labels = ["H (Mandante)", "D (Empate)", "A (Visitante)"]
    y_labels = [f"Under {YELL_LINE}", f"Over {YELL_LINE}"]
    c_labels = [f"Under {CORN_LINE}", f"Over {CORN_LINE}"]

    pred = {
        "match":       f"{home_team} vs {away_team} ({league})",
        "resultado": {
            "probs":      {l: round(float(p), 3) for l, p in zip(result_labels, pr)},
            "label":      result_labels[pr.argmax()],
            "confidence": round(float(pr.max()), 3),
        },
        "cartoes": {
            "probs":      {l: round(float(p), 3) for l, p in zip(cat_labels, py)},
            "label":      zip(y_labels, py),
            "confidence": round(float(py.max()), 3),
        },
        "escanteios": {
            "probs":      {l: round(float(p), 3) for l, p in zip(cat_labels, pc)},
            "label":      zip(c_labels, pc),
            "confidence": round(float(pc.max()), 3),
        },
    }
    return pred


def print_prediction(pred: dict):
    if "error" in pred:
        print(f"❌ Erro: {pred['error']}")
        return
    print(f"\n{'═'*55}")
    print(f"  🏟️  {pred['match']}")
    print(f"{'─'*55}")
    for alvo in ["resultado","cartoes","escanteios"]:
        p = pred[alvo]
        print(f"  {alvo.upper():12s} → {p['label']:20s} (confiança: {p['confidence']*100:.1f}%)")
        for label, prob in p["probs"].items():
            bar = "█" * int(prob * 25)
            print(f"    {label:18s} {prob*100:5.1f}%  {bar}")
    print(f"{'═'*55}")

print("Funções de predição prontas ✅")


Funções de predição prontas ✅


---
## Cenário 1 — Clássico do Brasileirão: Flamengo × Grêmio

**Perfil:** Flamengo como mandante forte (0.72), Grêmio visitante médio (0.59).  
**Esperado:** Vitória do Flamengo favorita, escanteios médio-alto.


In [ ]:
pred1 = predict_match("Flamengo", "Grêmio", "brasileirao_a", features_df)
print_prediction(pred1)

# Contexto histórico simulado
h2h_mask = (
    features_df["home_team"].isin(["Flamengo","Grêmio"]) &
    features_df["away_team"].isin(["Flamengo","Grêmio"]) &
    (features_df["league"] == "brasileirao_a")
)
h2h = features_df[h2h_mask].sort_values("date").tail(10)
print(f"\nÚltimos {len(h2h)} confrontos diretos disponíveis no dataset sintético")
print(h2h[["date","home_team","away_team","home_win_rate","away_win_rate"]].tail(5).to_string(index=False))


---
## Cenário 2 — Premier League: Manchester City × Leeds (Top vs Bottom)

**Perfil:** Campeão (0.76) contra time rebaixado (0.46).  
**Esperado:** Probabilidade altíssima de vitória City, xG elevado para o mandante.


In [ ]:
pred2 = predict_match("Manchester City", "Leeds", "premier_league", features_df)
print_prediction(pred2)


---
## Cenário 3 — La Liga: Real Madrid × Getafe (Rivalidade intensa, muitos cartões)

**Perfil:** Real Madrid favoritíssimo, Getafe conhecido por estilo físico.  
**Esperado:** Vitória do Real, cartões amarelos alto ou médio.


In [ ]:
pred3 = predict_match("Real Madrid", "Getafe", "la_liga", features_df)
print_prediction(pred3)


---
## Cenário 4 — Equilíbrio: Grêmio × Internacional (Gre-Nal)

**Perfil:** Dois times equilibrados, histórico intenso.  
**Esperado:** Resultado mais incerto, empate com probabilidade relevante.


In [ ]:
pred4 = predict_match("Grêmio", "Internacional", "brasileirao_a", features_df)
print_prediction(pred4)


---
## Cenário 5 — Liga Ofensiva: Liverpool × Manchester City

**Perfil:** Dois dos maiores xG do campeonato. Ambos com muitos escanteios.  
**Esperado:** Escanteios alto, gols esperados elevados, resultado incerto.


In [ ]:
pred5 = predict_match("Liverpool", "Manchester City", "premier_league", features_df)
print_prediction(pred5)


## Sumário dos 5 Cenários

In [ ]:
import pandas as pd

rows = []
for pred in [pred1, pred2, pred3, pred4, pred5]:
    if "error" in pred: continue
    rows.append({
        "Partida":          pred["match"],
        "Resultado":        pred["resultado"]["label"],
        "Conf. Resultado":  f"{pred['resultado']['confidence']*100:.1f}%",
        "Cartões":          pred["cartoes"]["label"],
        "Escanteios":       pred["escanteios"]["label"],
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))


---
## Próximos Passos

1. **`tools.py`** — adaptar `run_prediction_engine()` para usar este predictor  
2. **`orchestrator.py`** — agentic loop: LLM decide quais tools chamar  
3. **`llm_client.py`** — wrapper da Claude API  
4. **API Flask** — conectar tudo via `/api/chat`  
5. **Frontend React** — `PredictionCard` renderizado quando NN é acionada
